In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
from zen_garden.postprocess.results import Results

# === Dataset Setup ===
dataset_name_1 = 'Data_WT'
dataset_name_2 = 'Data_WT_new'
output_path_1 = os.path.join("outputs", dataset_name_1)
output_path_2 = os.path.join("outputs", dataset_name_2)
r1 = Results(output_path_1)
r2 = Results(output_path_2)

# === Output Save Directory (optional saving toggle) ===
save_plots = True
save_path = Path(r"C:\Users\nicol\OneDrive\Desktop\Semester_project\Plots_and_Figures\SCM_plots\wind_comparison")
if save_plots:
    os.makedirs(save_path, exist_ok=True)

# === Plotting Style ===
start_year = 2021
plt.rcParams.update({'font.size': 12})

marker_styles = {
    dataset_name_1: 'o',
    dataset_name_2: 's'
}

colors = {
    dataset_name_1: '#1f77b4',
    dataset_name_2: '#ff7f0e'
}

linestyles = {
    dataset_name_1: '--',
    dataset_name_2: '-'
}

line_widths = {
    dataset_name_1: 1.5,
    dataset_name_2: 1.5
}


# === Generic Time Series Plotting Function ===
def plot_time_series_styled(df1, df2, value_name, ylabel, title_prefix, file_prefix, dataset_name_1, dataset_name_2, save_path, save_plots=True):
    scenarios = df1["scenario"].unique()

    for scenario in scenarios:
        df1_s = df1[df1["scenario"] == scenario]
        df2_s = df2[df2["scenario"] == scenario]

        time_cols = [col for col in df1_s.columns if isinstance(col, (int, float))]
        times = time_cols

        df1_grouped = df1_s.groupby(df1_s[value_name])[times].sum()
        df2_grouped = df2_s.groupby(df2_s[value_name])[times].sum()

        entities = sorted(set(df1_grouped.index).union(df2_grouped.index))
        non_zero = [
            entity for entity in entities
            if df1_grouped.loc[entity].sum() > 0 or df2_grouped.loc[entity].sum() > 0
        ]

        for entity in non_zero:
            plt.figure(figsize=(10, 5))

            for label, grouped in zip([dataset_name_1, dataset_name_2], [df1_grouped, df2_grouped]):
                if entity in grouped.index:
                    plt.plot(
                        times, grouped.loc[entity],
                        marker=marker_styles[label],
                        linestyle=linestyles[label],
                        linewidth=line_widths[label],
                        color=colors[label],
                        label=label.replace("_", " ")
                    )

            title = f"{title_prefix}"
            plt.title(title, fontsize=14)
            plt.xlabel("Year")
            plt.ylabel(ylabel)
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.legend(loc='upper left')
            plt.tight_layout()

            xticks_labels = [str(2021 + i) for i in range(len(times))]
            plt.xticks(ticks=times, labels=xticks_labels)

            if save_plots:
                for fmt in ['pdf', 'png']:
                    filename = f"{file_prefix}_{value_name}_{entity}_scenario_{scenario}.{fmt}".replace(" ", "_").lower()
                    plt.savefig(save_path / filename, format=fmt, dpi=300, bbox_inches='tight')

            plt.show()


# === Cost Plot Function ===
def plot_cost_total(cost1, cost2, ylabel="Total Cost [MEUR]", title="Total Cost", file_prefix="cost_total"):
    time_cols = [col for col in cost1.columns if isinstance(col, (int, float))]
    times = time_cols

    if "carrier" in cost1.columns and "carrier" in cost2.columns:
        cost1_grouped = cost1.groupby("carrier")[times].sum()
        cost2_grouped = cost2.groupby("carrier")[times].sum()
        carriers = sorted(set(cost1_grouped.index).union(cost2_grouped.index))
    else:
        cost1_grouped = cost1[time_cols].sum().to_frame().T
        cost2_grouped = cost2[time_cols].sum().to_frame().T
        carriers = ["total"]

    for carrier in carriers:
        plt.figure(figsize=(10, 5))

        for label, grouped in zip([dataset_name_1, dataset_name_2], [cost1_grouped, cost2_grouped]):
            if carrier in grouped.index:
                data = grouped.loc[carrier]
            else:
                data = grouped.iloc[0]

            plt.plot(
                times, data,
                marker=marker_styles[label],
                linestyle=linestyles[label],
                linewidth=line_widths[label],
                color=colors[label],
                label=label.replace("_", " ")
            )

        plt.title(f"{title} — Carrier: {carrier}", fontsize=14)
        plt.xlabel("Year")
        plt.ylabel(ylabel)
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.legend(loc='upper left')
        plt.tight_layout()
        xticks_positions = times
        xticks_labels = [str(start_year + i) for i in range(len(xticks_positions))]
        plt.xticks(ticks=xticks_positions, labels=xticks_labels)

        if save_plots:
            filename = f"{file_prefix}_{carrier}.pdf".replace(" ", "_").lower()
            plt.savefig(save_path / filename, format='pdf', bbox_inches='tight')

        plt.show()


# === Load and Preprocess Data ===
demand1 = r1.get_full_ts("demand").reset_index().rename(columns={"level_0": "scenario"})
demand2 = r2.get_full_ts("demand").reset_index().rename(columns={"level_0": "scenario"})

capacity_full_1 = r1.get_full_ts("capacity").reset_index().rename(columns={"level_0": "scenario"})
capacity_full_2 = r2.get_full_ts("capacity").reset_index().rename(columns={"level_0": "scenario"})

cost1 = r1.get_full_ts("cost_total").reset_index()
cost2 = r2.get_full_ts("cost_total").reset_index()

capex1 = r1.get_full_ts("cost_capex_yearly_total").reset_index()
capex2 = r2.get_full_ts("cost_capex_yearly_total").reset_index()

opex1 = r1.get_full_ts("cost_opex_yearly_total").reset_index()
opex2 = r2.get_full_ts("cost_opex_yearly_total").reset_index()

capacity1 = r1.get_full_ts("capacity").reset_index().rename(columns={"level_0": "scenario"})
capacity2 = r2.get_full_ts("capacity").reset_index().rename(columns={"level_0": "scenario"})

capacity_add_1 = r1.get_full_ts("capacity_addition").reset_index().rename(columns={"level_0": "scenario"})
capacity_add_2 = r2.get_full_ts("capacity_addition").reset_index().rename(columns={"level_0": "scenario"})


# === Filter: Turbine capacity by location ===
capacity_turbine_loc_1 = capacity_full_1[
    (capacity_full_1["technology"] == "Turbine") &
    (capacity_full_1["capacity_type"] == "power")
    ]

capacity_turbine_loc_2 = capacity_full_2[
    (capacity_full_2["technology"] == "Turbine") &
    (capacity_full_2["capacity_type"] == "power")
    ]

print("Filtered turbine capacity (by location) — Dataset 1:", capacity_turbine_loc_1.shape)
print("Filtered turbine capacity (by location) — Dataset 2:", capacity_turbine_loc_2.shape)

# === Plot Calls ===

# === Plot Turbine Demand (no scenario grouping) ===
def plot_demand_noscenario(df1, df2, value_name, ylabel, title, file_prefix):
    time_cols = [col for col in df1.columns if isinstance(col, (int, float))]
    times = time_cols

    df1_grouped = df1.groupby(value_name)[time_cols].sum()
    df2_grouped = df2.groupby(value_name)[time_cols].sum()

    entities = sorted(set(df1_grouped.index).union(df2_grouped.index))
    non_zero = [
        entity for entity in entities
        if df1_grouped.loc[entity].sum() > 0 or df2_grouped.loc[entity].sum() > 0
    ]

    for entity in non_zero:
        plt.figure(figsize=(10, 5))

        for label, grouped in zip([dataset_name_1, dataset_name_2], [df1_grouped, df2_grouped]):
            if entity in grouped.index:
                plt.plot(
                    times, grouped.loc[entity],
                    marker=marker_styles[label],
                    linestyle=linestyles[label],
                    linewidth=line_widths[label],
                    color=colors[label],
                    label=label.replace("_", " ")
                )

        plt.title(f"{title} — {value_name.capitalize()}: {entity}", fontsize=14)
        plt.xlabel("Year")
        plt.ylabel(ylabel)
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.legend(loc='upper left')
        plt.tight_layout()
        xticks_labels = [str(start_year + i) for i in range(len(times))]
        plt.xticks(ticks=times, labels=xticks_labels)

        if save_plots:
            filename = f"{file_prefix}_{value_name}_{entity}.pdf".replace(" ", "_").lower()
            plt.savefig(save_path / filename, format='pdf', bbox_inches='tight')

        plt.show()

def plot_cost_total_billion(cost1, cost2, ylabel="Total Cost [B€]", title="Wind Turbine Supply Chain Cost", file_prefix="cost_total"):
    time_cols = [col for col in cost1.columns if isinstance(col, (int, float))]
    times = time_cols

    # Aggregate: use carrier if available, else total
    if "carrier" in cost1.columns and "carrier" in cost2.columns:
        cost1_grouped = cost1.groupby("carrier")[times].sum()
        cost2_grouped = cost2.groupby("carrier")[times].sum()
        carriers = sorted(set(cost1_grouped.index).union(cost2_grouped.index))
    else:
        cost1_grouped = cost1[time_cols].sum().to_frame().T
        cost2_grouped = cost2[time_cols].sum().to_frame().T
        carriers = ["total"]

    for carrier in carriers:
        plt.figure(figsize=(10, 5))

        for label, grouped in zip([dataset_name_1, dataset_name_2], [cost1_grouped, cost2_grouped]):
            if carrier in grouped.index:
                data = grouped.loc[carrier]
            else:
                data = grouped.iloc[0]

            # ✅ Convert from M€ to B€ (divide by 1000)
            data_billion = data / 1_000

            plt.plot(
                times, data_billion,
                marker=marker_styles[label],
                linestyle=linestyles[label],
                linewidth=line_widths[label],
                color=colors[label],
                label=label.replace("_", " ")
            )

        plt.title(title, fontsize=14)
        plt.xlabel("Year")
        plt.ylabel(ylabel)
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.legend(loc='upper left')
        plt.tight_layout()
        xticks_labels = [str(start_year + i) for i in range(len(times))]
        plt.xticks(ticks=times, labels=xticks_labels)

        # Format y-axis with commas and one decimal
        ax = plt.gca()
        ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))

        if save_plots:
            filename = f"{file_prefix}_{carrier}.pdf".replace(" ", "_").lower()
            plt.savefig(save_path / filename, format='pdf', bbox_inches='tight')

        plt.show()



def plot_total_time_series(df1, df2, time_label, ylabel, title, file_prefix):
    time_cols = [col for col in df1.columns if isinstance(col, (int, float))]
    times = time_cols

    total1 = df1[time_cols].sum()
    total2 = df2[time_cols].sum()

    plt.figure(figsize=(10, 5))
    for label, total in zip([dataset_name_1, dataset_name_2], [total1, total2]):
        plt.plot(
            times, total,
            marker=marker_styles[label],
            linestyle=linestyles[label],
            linewidth=line_widths[label],
            color=colors[label],
            label=label.replace("_", " ")
        )

    plt.title(title, fontsize=14)
    plt.xlabel(time_label)
    plt.ylabel(ylabel)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend(loc='upper left')
    plt.tight_layout()
    xticks_labels = [str(start_year + i) for i in range(len(times))]
    plt.xticks(ticks=times, labels=xticks_labels)

    if save_plots:
        filename = f"{file_prefix}.pdf".replace(" ", "_").lower()
        plt.savefig(save_path / filename, format='pdf', bbox_inches='tight')

    plt.show()


# === Filter Turbine carrier only ===
demand_turbine_1 = demand1[demand1["carrier"].str.lower() == "turbine"]
demand_turbine_2 = demand2[demand2["carrier"].str.lower() == "turbine"]

# === Print Turbine Demand by Node and Year in Text Form ===
nodes_of_interest = ["DEU", "CHE", "DNK", "SWE", "GBR", "NLD", "ROE"]
time_cols = [col for col in demand_turbine_1.columns if isinstance(col, (int, float))]

# Aggregate by node and year
demand_by_node_1 = (
    demand_turbine_1[demand_turbine_1["node"].isin(nodes_of_interest)]
    .groupby("node")[time_cols]
    .sum()
    .reindex(nodes_of_interest)
)

demand_by_node_2 = (
    demand_turbine_2[demand_turbine_2["node"].isin(nodes_of_interest)]
    .groupby("node")[time_cols]
    .sum()
    .reindex(nodes_of_interest)
)

# Compute total demand per year
total_demand_1 = demand_by_node_1.sum()
total_demand_2 = demand_by_node_2.sum()

# === Total Turbine Demand Summed Over All Years ===
sum_total_demand_1 = total_demand_1.sum()
sum_total_demand_2 = total_demand_2.sum()

print("\n=== Total Turbine Demand Summed Over All Years ===")
print(f"{dataset_name_1}: {sum_total_demand_1:,.2f} GWh")
print(f"{dataset_name_2}: {sum_total_demand_2:,.2f} GWh")


# Combine into one table
combined_demand = pd.concat(
    [demand_by_node_1.add_suffix(" (WT)"), demand_by_node_2.add_suffix(" (WT_new)")],
    axis=1
)

# Add a TOTAL row
combined_demand.loc["TOTAL"] = list(total_demand_1) + list(total_demand_2)

# Save to CSV
csv_output_path = save_path / "turbine_demand_by_node_and_total.csv"
combined_demand.to_csv(csv_output_path)

# Print summary
print("\n=== Turbine Demand by Node and Year ===\n")
for node in nodes_of_interest + ["TOTAL"]:
    print(f"\nNode: {node}")
    for year in time_cols:
        val1 = demand_by_node_1.loc[node, year] if node in demand_by_node_1.index else total_demand_1[year]
        val2 = demand_by_node_2.loc[node, year] if node in demand_by_node_2.index else total_demand_2[year]
        print(f"  {year}: WT = {val1:.2f} GWh, WT_new = {val2:.2f} GWh")


# Plot demand by carrier with styling
plot_time_series_styled(
    df1=demand1,
    df2=demand2,
    value_name="carrier",
    ylabel="Demand [GWh]",
    title_prefix="Turbine Demand",
    file_prefix="turbine_demand",
    dataset_name_1=dataset_name_1,
    dataset_name_2=dataset_name_2,
    save_path=save_path,
    save_plots=save_plots
)

# Plot capacity by location with styling
plot_time_series_styled(
    df1=capacity1,
    df2=capacity2,
    value_name="location",
    ylabel="Capacity Addition [GW]",
    title_prefix="Turbine Capacity Addition",
    file_prefix="turbine_capacity_addition",
    dataset_name_1=dataset_name_1,
    dataset_name_2=dataset_name_2,
    save_path=save_path,
    save_plots=save_plots
)





plot_total_time_series(
    df1=demand_turbine_1,
    df2=demand_turbine_2,
    time_label="Year",
    ylabel="Turbine Demand [GWh]",
    title="Total Wind Turbine Demand",
    file_prefix="turbine_demand_total"
)

plot_cost_total_billion(
    cost1=cost1,
    cost2=cost2,
    ylabel="Total Cost [Billion €]",
    title="Wind Turbine Supply Chain Cost",
    file_prefix="cost_total"
)


# Cost total
plot_cost_total(
    cost1=cost1,
    cost2=cost2,
    ylabel="Total Cost [MEUR]",
    title="Total System Cost",
    file_prefix="cost_total"
)
